# MTL — TEyeD Dikablis
Obavezne konfiguracije, etapno treniranje, Cross-Feature Fusion. `RUN_MODE="minimal"` pokreće 6 treniranja, `RUN_MODE="mandatory"` svih 9. Vremenske procjene su okvirne.

In [1]:
# PROFOVE TOČKE POKRIVENE OVOM SKRIPTOM:
#   Točka 3  – kutna pogreška (arccos) u stupnjevima
#   Točka 4  – usporedba s gaze_only (runs/gaze_only/) i trivijalnim baselineom
#   Točka 6  – mtl_wseg0 vs mtl_full
#   Točka 7  – 3 seeda za obavezne konfiguracije, mean ± std
#   Točka 8  – Wilcoxon po ispitaniku (subject_id)
#   Točka 10 – Cross-Feature Fusion + skip veze
# MINIMALNI SET ZA PROFOVE TOČKE:
#   mtl_full × 3 + mtl_wseg0 × 3 = 6 treniranja (~10 h; procjena)
# PREPORUČENI SET:
#   + mtl_pupil × 3 = 9 treniranja (~20 h; procjena)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'
import sys, gc, json, time, math, random, warnings, traceback, subprocess
from pathlib import Path
import numpy as np, pandas as pd
sys.modules['jax']=None
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
BASE_PATH=Path('/home/fipulab/anastazija/TEyeD_preprocessed')
ROOT=BASE_PATH/'128x128_stride10_q4_seg_pupil-iris-lid_2D'
RUNS=Path('/home/fipulab/anastazija/runs')
OUT=RUNS/'mtl'; OUT.mkdir(parents=True,exist_ok=True)
PARTS=['pupil','iris','lid']
MANDATORY_CONFIGS=[
 {'name':'mtl_full','seg_parts':['pupil','iris','lid'],'w_seg':0.2},
 {'name':'mtl_wseg0','seg_parts':['pupil','iris','lid'],'w_seg':0.0},
 {'name':'mtl_pupil','seg_parts':['pupil'],'w_seg':0.2},
]
MANDATORY_SEEDS=[42,123,2024]
SEEDS_MANDATORY=MANDATORY_SEEDS
# 'minimal' = 6 treniranja; 'mandatory' = svih 9. Bez drugih ablacija.
RUN_MODE='mandatory'
assert RUN_MODE in ('minimal','mandatory')
ACTIVE_CONFIGS=MANDATORY_CONFIGS
BATCH_SIZE=128
BATCH_FALLBACKS=[128,64,32,16]
IMAGE_SHAPE=(128,128,3)
W_GAZE=1.0
PHASE1_EPOCHS=5
PHASE2_EPOCHS=25
VAL_STEPS_LIMIT=100
TRAIN_SAMPLE_FRAC=0.5
USE_MIXED_PRECISION=True
NUM_PARALLEL_CALLS=4
UNFROZEN_LAYERS_PHASE2=30
EARLY_STOP_PATIENCE=4
REDUCE_LR_PATIENCE=1
try:
 tf.config.threading.set_inter_op_parallelism_threads(4)
 tf.config.threading.set_intra_op_parallelism_threads(4)
except RuntimeError as e: warnings.warn(str(e))
GPUS=tf.config.list_physical_devices('GPU')
for gpu in GPUS:
 try: tf.config.experimental.set_memory_growth(gpu,True)
 except RuntimeError as e: warnings.warn(str(e))
keras.mixed_precision.set_global_policy('mixed_float16' if GPUS and USE_MIXED_PRECISION else 'float32')
try:
 import psutil
 _HAS_PSUTIL=True
except ImportError:
 _HAS_PSUTIL=False
if GPUS:
 try:
  print(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total,memory.used,memory.free','--format=csv,noheader,nounits'],text=True))
 except Exception as e: warnings.warn(f'GPU memorija nije očitana: {e}')
else: warnings.warn('GPU nije dostupan')
if _HAS_PSUTIL:
 mem=psutil.virtual_memory()
 print(f'RAM slobodno: {mem.available/2**30:.2f} GB')
print('TensorFlow:',tf.__version__,'policy:',keras.mixed_precision.global_policy())
print('Pokretanje:',RUN_MODE,'treniranja:',len(ACTIVE_CONFIGS)*len(SEEDS_MANDATORY))


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3090, compute capability 8.6
24268, 24, 24244

RAM slobodno: 60.45 GB
TensorFlow: 2.11.0 policy: <Policy "mixed_float16">
Pokretanje: mandatory treniranja: 9


## 2. Provjera resursa i referentnih modela

In [2]:
metadata_path=ROOT/'metadata.json'
if metadata_path.exists():
 try:
  metadata=json.loads(metadata_path.read_text(encoding='utf-8'))
  print('metadata.json: OK; ključevi:',list(metadata)[:20])
 except Exception as e: warnings.warn(f'Neispravan metadata.json: {e}')
else: warnings.warn(f'metadata.json nije pronađen: {metadata_path}')
for seed in SEEDS_MANDATORY:
 seg=RUNS/'seg_only'/f'seed_{seed}'/'best_parts.keras'
 if not seg.exists(): warnings.warn(f'Nedostaje predtrenirani segmentacijski model: {seg}')
 gaze=RUNS/'gaze_only'/f'seed_{seed}'/'test_predictions.csv'
 if not gaze.exists(): warnings.warn(f'Nedostaje gaze_only referenca: {gaze}')
trivial=RUNS/'trivial_baseline'/'test_predictions.csv'
if not trivial.exists(): warnings.warn(f'Nedostaje trivijalni baseline: {trivial}')


/tmp/ipykernel_371991/2942511861.py:7: UserWarning: metadata.json nije pronađen: /home/fipulab/anastazija/TEyeD_preprocessed/128x128_stride10_q4_seg_pupil-iris-lid_2D/metadata.json
  else: warnings.warn(f'metadata.json nije pronađen: {metadata_path}')
/tmp/ipykernel_371991/2942511861.py:14: UserWarning: Nedostaje trivijalni baseline: /home/fipulab/anastazija/runs/trivial_baseline/test_predictions.csv
  if not trivial.exists(): warnings.warn(f'Nedostaje trivijalni baseline: {trivial}')


## 3. Učitavanje podataka i provjera

In [3]:
DF={}
required={'filename','x','y','z','subject_id','video_id',*[f'mask_valid_{p}' for p in PARTS]}
for split in ('train','val','test'):
 root=ROOT/split; path=root/'labels.csv'
 if not path.exists(): raise FileNotFoundError(path)
 df=pd.read_csv(path); missing=required-set(df.columns)
 if missing: raise ValueError(f'{path}: nedostaju {sorted(missing)}')
 for col in ('x','y','z'): df[col]=pd.to_numeric(df[col],errors='coerce')
 gaze=df[['x','y','z']].to_numpy(np.float32); norm=np.linalg.norm(gaze,axis=1)
 good=np.isfinite(gaze).all(axis=1)&np.isfinite(norm)&(norm>0)
 if (~good).any(): warnings.warn(f'{split}: izostavljam {(~good).sum()} nevaljanih gaze vektora')
 df=df.loc[good].copy().reset_index(drop=True)
 gaze=gaze[good]/norm[good,None]; df[['x','y','z']]=gaze
 for part in PARTS:
  if not (root/f'seg_{part}_2D').is_dir(): raise FileNotFoundError(root/f'seg_{part}_2D')
 DF[split]=df
 print(f'{split}: {len(df):,} uzoraka, {df.video_id.nunique()} videa, {df.subject_id.nunique()} ispitanika')
for a,b in [('train','val'),('train','test'),('val','test')]:
 overlap=set(DF[a].subject_id.astype(str))&set(DF[b].subject_id.astype(str))
 if overlap: raise ValueError(f'Subject leakage {a}/{b}: {list(overlap)[:10]}')
print('Subject-disjoint: OK')
for seed in SEEDS_MANDATORY:
 seg=RUNS/'seg_only'/f'seed_{seed}'/'best_parts.keras'
 if not seg.exists(): warnings.warn(f'Nedostaje seg model: {seg}; taj seed neće moći trenirati.')
 baseline=RUNS/'gaze_only'/f'seed_{seed}'/'test_predictions.csv'
 if not baseline.exists(): warnings.warn(f'Nedostaje gaze-only predikcija: {baseline}; Wilcoxon za seed preskačem.')

train: 339,466 uzoraka, 264 videa, 76 ispitanika
val: 135,958 uzoraka, 108 videa, 22 ispitanika
test: 79,964 uzoraka, 50 videa, 11 ispitanika
Subject-disjoint: OK


## 4. tf.data i augmentacija

In [4]:
# [RAM zaštita] R5: ograničen prefetch na 2 batcha.
def seed_everything(seed):
 random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def project(image,mask,t):
 shape=tf.constant([128,128]); t=t[None,:]
 image=tf.raw_ops.ImageProjectiveTransformV3(images=image[None],transforms=t,output_shape=shape,interpolation='BILINEAR',fill_mode='REFLECT',fill_value=0.)[0]
 mask=tf.raw_ops.ImageProjectiveTransformV3(images=mask[None],transforms=t,output_shape=shape,interpolation='NEAREST',fill_mode='REFLECT',fill_value=0.)[0]
 return image,mask

def augment(image,mask,gaze):
 flip=tf.random.uniform(())<.5
 image=tf.cond(flip,lambda:tf.image.flip_left_right(image),lambda:image)
 mask=tf.cond(flip,lambda:tf.image.flip_left_right(mask),lambda:mask)
 gaze=tf.cond(flip,lambda:tf.stack([-gaze[0],gaze[1],gaze[2]]),lambda:gaze)
 angle=tf.random.uniform((),-.025,.025)*(2.*math.pi)
 c=tf.cos(angle); s=tf.sin(angle); cx=tf.constant(63.5)
 t=tf.stack([c,s,cx-c*cx-s*cx,-s,c,cx+s*cx-c*cx,0.,0.])
 image,mask=project(image,mask,t)
 gaze=tf.math.l2_normalize(tf.stack([c*gaze[0]-s*gaze[1],s*gaze[0]+c*gaze[1],gaze[2]]))
 shear=tf.random.uniform((),-.02,.02)
 t=tf.stack([1.,shear,-shear*63.5,0.,1.,0.,0.,0.])
 image,mask=project(image,mask,t)
 gaze=tf.math.l2_normalize(tf.stack([gaze[0]+shear*gaze[1],gaze[1],gaze[2]]))
 zoom=tf.random.uniform((),.95,1.05); z=1./zoom; shift=63.5-z*63.5
 t=tf.stack([z,0.,shift,0.,z,shift,0.,0.]); image,mask=project(image,mask,t)
 image=tf.image.random_brightness(image,max_delta=.08)
 image=tf.image.random_contrast(image,lower=.9,upper=1.1)
 return tf.clip_by_value(image,-1.,1.),mask,gaze

def make_dataset(df,split,batch,training=False,seed=42):
 names=df.filename.astype(str).to_numpy()
 gaze=df[['x','y','z']].to_numpy(np.float32)
 valid=df[[f'mask_valid_{p}' for p in PARTS]].fillna(0).to_numpy(np.float32)
 ds=tf.data.Dataset.from_tensor_slices((names,gaze,valid))
 if training: ds=ds.shuffle(min(len(df),20000),seed=seed,reshuffle_each_iteration=True)
 root=str(ROOT/split)
 def load(filename,gaze,valid):
  image=tf.io.decode_jpeg(tf.io.read_file(tf.strings.join([root,'/',filename])),channels=3)
  image=tf.image.resize(image,[128,128]); image=tf.cast(image,tf.float32)/127.5-1.
  pieces=tf.strings.split(filename,'/'); video=pieces[-2]
  frame=tf.strings.regex_replace(pieces[-1],r'\.jpg$','.png')
  masks=[]
  for part in PARTS:
   path=tf.strings.join([root,'/seg_',part,'_2D/',video,'/',frame])
   m=tf.io.decode_png(tf.io.read_file(path),channels=1)
   m=tf.image.resize(m,[128,128],method='nearest'); masks.append(tf.cast(m>0,tf.float32))
  mask=tf.concat(masks,axis=-1)
  if training: image,mask,gaze=augment(image,mask,gaze)
  # Validnost se čuva odvojeno od augmentiranih maski.
  target=tf.concat([mask,tf.ones_like(mask)*tf.reshape(valid,[1,1,3])],axis=-1)
  image.set_shape([128,128,3]); target.set_shape([128,128,6]); gaze.set_shape([3])
  return image,{'seg':target,'gaze_output':gaze}
 # [M1] Cache samo val/test izvornih putanja i oznaka, prije skupog dekodiranja.
 if not training: ds=ds.cache()
 ds=ds.map(load,num_parallel_calls=NUM_PARALLEL_CALLS,deterministic=not training)
 # [R5] MTL batch može zauzeti ~75 MB; AUTOTUNE može zadržati 10+ batcheva u host RAM-u.
 # Fiksni prefetch(2) ograničava memoriju i održava GPU opskrbljenim podacima.
 return ds.batch(batch).prefetch(2)

## 5. Arhitektura, gubitci i metrike

In [5]:
# [RAM zaštita] R2: EpochTimer prati host RAM.
@keras.utils.register_keras_serializable(package='MTL')
def unit_gaze(x): return tf.math.l2_normalize(tf.cast(x,tf.float32),axis=-1)

@keras.utils.register_keras_serializable(package='MTL')
def gaze_cosine(y_true,y_pred):
 y_true=unit_gaze(y_true); y_pred=unit_gaze(y_pred)
 return 1.-tf.reduce_sum(y_true*y_pred,axis=-1)

@keras.utils.register_keras_serializable(package='MTL')
def mean_angle_error(y_true,y_pred):
 dot=tf.reduce_sum(unit_gaze(y_true)*unit_gaze(y_pred),axis=-1)
 return tf.acos(tf.clip_by_value(dot,-1.,1.))*180./math.pi

@keras.utils.register_keras_serializable(package='MTL')
class MaskedBCE(keras.losses.Loss):
 def __init__(self,parts,name='masked_bce',**kwargs): super().__init__(name=name,**kwargs); self.parts=list(parts)
 def call(self,y_true,y_pred):
  ids=[PARTS.index(p) for p in self.parts]
  truth=tf.gather(y_true[...,:3],ids,axis=-1)
  valid=tf.gather(y_true[...,3:],ids,axis=-1)
  pred=tf.gather(tf.cast(y_pred,tf.float32),ids,axis=-1)
  per=keras.backend.binary_crossentropy(truth,pred)*valid
  # Keras reducira prosjek po batchu; svaki uzorak normaliziran po valjanim pikselima.
  return tf.reduce_sum(per,axis=[1,2,3])/tf.maximum(tf.reduce_sum(tf.broadcast_to(valid,tf.shape(per)),axis=[1,2,3]),1.)
 def get_config(self): return {**super().get_config(),'parts':self.parts}

@keras.utils.register_keras_serializable(package='MTL')
class PartIoU(keras.metrics.Metric):
 def __init__(self,part,**kwargs):
  super().__init__(name='iou_'+part,**kwargs); self.part=part; self.idx=PARTS.index(part)
  self.inter=self.add_weight(name='intersection',initializer='zeros'); self.union=self.add_weight(name='union',initializer='zeros')
 def update_state(self,y_true,y_pred,sample_weight=None):
  i=self.idx; valid=y_true[...,i+3]>.5; truth=tf.logical_and(y_true[...,i]>.5,valid)
  pred=tf.logical_and(tf.cast(y_pred[...,i],tf.float32)>.5,valid)
  self.inter.assign_add(tf.reduce_sum(tf.cast(truth&pred,tf.float32)))
  self.union.assign_add(tf.reduce_sum(tf.cast(truth|pred,tf.float32)))
 def result(self): return tf.math.divide_no_nan(self.inter,self.union)
 def reset_state(self): self.inter.assign(0.); self.union.assign(0.)
 def get_config(self): return {**super().get_config(),'part':self.part}

def build_mtl(seed):
 path=RUNS/'seg_only'/f'seed_{seed}'/'best_parts.keras'
 if not path.exists(): raise FileNotFoundError(f'Nedostaje predtrenirani seg model: {path}')
 # [BN GPU determinism] Rekonstruiraj samo BN operacije kao non-fused.
 # FusedBatchNormGradV3 za zamrznuti BN nema deterministički GPU gradijent
 # u TensorFlow verziji na serveru. Iste BN težine/statistike i arhitektura.
 original_seg=keras.models.load_model(path,compile=False)
 original_encoder=original_seg.get_layer('feature_extractor')
 def clone_encoder_layer(layer):
  cfg=layer.get_config()
  if isinstance(layer,layers.BatchNormalization):
   if 'fused' in cfg: cfg['fused']=False
  return layer.__class__.from_config(cfg)
 nonfused_encoder=keras.models.clone_model(original_encoder,clone_function=clone_encoder_layer)
 def clone_seg_layer(layer):
  if layer.name=='feature_extractor': return nonfused_encoder
  return layer.__class__.from_config(layer.get_config())
 seg=keras.models.clone_model(original_seg,clone_function=clone_seg_layer)
 seg.set_weights(original_seg.get_weights())
 # TensorFlow/Keras 2.x može automatski vratiti fused=True pri build() za 4D BN,
 # čak i ako je clone_model dobio fused=False u konfiguraciji.
 # Postavi oba atributa na False na već izgrađenim BN slojevima.
 # Ovo ne mijenja BN težine, moving statistics niti trainable stanje.
 for bn in nonfused_encoder.submodules:
  if isinstance(bn,layers.BatchNormalization):
   bn.fused=False
   if hasattr(bn,'_fused'): bn._fused=False
   if getattr(bn,'fused',False) or getattr(bn,'_fused',False):
    raise RuntimeError(f'Ne mogu isključiti fused BatchNormalization: {bn.name}')
 print('BN determinism: rekonstruiran encoder s non-fused BatchNormalization; težine sačuvane.')
 if seg.output_shape[-1]!=3: raise ValueError(f'{path}: očekivana 3 segmentacijska kanala')
 # Dijeljeni enkoder i U-Net dekoder zadržavaju točno učitane težine.
 encoder=seg.get_layer('feature_extractor')
 # Oslobodi izvorni model nakon kopiranja svih težina; novi seg/encoder ostaju živi.
 del original_seg, original_encoder
 gc.collect()
 # [K1] Traži posljednji trokanalni Conv2D i posljednji upsampling prije njega.
 seg_convs=[(i,l) for i,l in enumerate(seg.layers) if isinstance(l,layers.Conv2D) and l.filters==3]
 if not seg_convs: raise ValueError('Nema završnog Conv2D(filters=3). Slojevi: '+str([(l.name,type(l).__name__,getattr(l,'output_shape',str(getattr(l,'output',None).shape if hasattr(l,'output') else '?'))) for l in seg.layers]))
 seg_conv_idx,seg_conv=seg_convs[-1]
 candidates=[l for l in seg.layers[:seg_conv_idx] if isinstance(l,(layers.Conv2DTranspose,layers.UpSampling2D,layers.Resizing))]
 if not candidates:
  details=[(l.name,type(l).__name__,str(getattr(getattr(l,'output',None),'shape',getattr(l,'output_shape','?')))) for l in seg.layers]
  raise ValueError(f'Nisu pronađene dekoderske značajke prije {seg_conv.name}; svi slojevi: {details}')
 decoder_layer=candidates[-1]; decoder_features=decoder_layer.output
 print('decoder_features:',decoder_layer.name,decoder_features.shape)
 # [K2] Nested encoder: uzmi izlaz poziva u grafu segmentacijskog modela.
 # Keras 3 nema get_output_at; izlazi nested sloja u seg grafu su encoder.output.
 try:
  enc_outputs=encoder.get_output_at(-1)
 except (AttributeError,TypeError):
  enc_outputs=encoder.output
 outputs=list(enc_outputs) if isinstance(enc_outputs,(list,tuple)) else [enc_outputs]
 spatial=[]
 for tensor in outputs:
  shape=tensor.shape
  if len(shape)!=4: continue
  h,w=shape[1],shape[2]
  if h is not None and w is not None: spatial.append((int(h)*int(w),tensor))
 if not spatial: raise ValueError(f'Enkoder nema 4D izlaz s poznatim H,W: {[str(t.shape) for t in outputs]}')
 bottleneck=min(spatial,key=lambda item:item[0])[1]
 h,w=bottleneck.shape[1:3]
 if int(h)>=128 or int(w)>=128:
  raise ValueError(f'Bottleneck nije smanjen ispod 128: {[str(t.shape) for t in outputs]}')
 print('encoder_features:',encoder.name, bottleneck.shape)
 seg_gap=layers.GlobalAveragePooling2D(name='fusion_seg_gap')(decoder_features)
 enc_gap=layers.GlobalAveragePooling2D(name='fusion_encoder_gap')(bottleneck)
 fused=layers.Concatenate(name='fusion_concat')([enc_gap,seg_gap])
 fused=layers.Dense(128,activation='relu',name='fusion_dense1')(fused)
 fused=layers.Dropout(.3,name='fusion_dropout')(fused)
 fused=layers.Dense(64,activation='relu',name='fusion_dense2')(fused)
 raw=layers.Dense(3,name='gaze_raw',dtype='float32')(fused)
 gaze=layers.Lambda(unit_gaze,name='gaze_output',dtype='float32')(raw)
 # Gaze loss propagira gradijent kroz fusion_seg_gap i U-Net dekoder čak i kad je w_seg=0.
 model=keras.Model(seg.input,{'seg':seg.output,'gaze_output':gaze},name='MTL_CrossFeatureFusion')
 return model,encoder

def optimizer(phase):
 if phase==1: cls=keras.optimizers.Adam; kwargs={'learning_rate':1e-3}
 else: cls=getattr(keras.optimizers,'AdamW',tf.keras.optimizers.experimental.AdamW); kwargs={'learning_rate':1e-4,'weight_decay':1e-3}
 try: return cls(jit_compile=False,**kwargs)
 except TypeError: return cls(**kwargs)

def compile_model(model,parts,w_seg,phase):
 model.compile(optimizer=optimizer(phase),loss={'seg':MaskedBCE(parts),'gaze_output':gaze_cosine},
  loss_weights={'seg':float(w_seg),'gaze_output':W_GAZE},
  metrics={'seg':[PartIoU(p) for p in PARTS],'gaze_output':[mean_angle_error]},jit_compile=False)

class EpochTimer(keras.callbacks.Callback):
 def __init__(self,limit_hours=6): super().__init__(); self.start=None; self.durations=[]; self.limit=limit_hours
 def on_epoch_begin(self,epoch,logs=None): self.start=time.monotonic()
 def on_epoch_end(self,epoch,logs=None):
  self.durations.append(time.monotonic()-self.start)
  print(f'Vrijeme epohe {self.durations[-1]/60:.1f} min; prosjek {np.mean(self.durations)/60:.1f} min')
  if sum(self.durations)>self.limit*3600: warnings.warn('Preko 6 h: razmotri manje epoha ili batch size.')
  # [R2] Praćenje host RAM-a na kraju svake epohe bez ponovnog importa psutil.
  if _HAS_PSUTIL:
   try:
    ram=psutil.virtual_memory()
    print(f'RAM: {ram.percent:.1f}% zauzeto, slobodno {ram.available/2**30:.2f} GB')
    if ram.percent>90:
     warnings.warn(f'RAM {ram.percent:.1f}% — visok rizik od OOM killera. '
                   'Razmotri prekid i smanjenje batch sizea.')
   except Exception as e:
    warnings.warn(f'RAM provjera u EpochTimer nije uspjela: {e}')

# [V3] Faze imaju odvojene checkpointove, bez prepisivanja faze 1.
def callbacks(out,phase):
 if phase not in (1,2): raise ValueError(f'Nepoznata faza: {phase}')
 monitor='val_gaze_output_mean_angle_error'
 return [keras.callbacks.ModelCheckpoint(str(out/f'best_phase{phase}.weights.h5'),monitor=monitor,mode='min',save_best_only=True,save_weights_only=True),
  keras.callbacks.ReduceLROnPlateau(monitor=monitor,mode='min',factor=.5,patience=REDUCE_LR_PATIENCE,min_lr=1e-6),
  keras.callbacks.EarlyStopping(monitor=monitor,mode='min',patience=EARLY_STOP_PATIENCE,restore_best_weights=True),EpochTimer()]

## 6. Treniranje jedne kombinacije

In [6]:
def sample_train(df,seed):
 if not 0<TRAIN_SAMPLE_FRAC<=1: raise ValueError('TRAIN_SAMPLE_FRAC mora biti u (0,1]')
 if TRAIN_SAMPLE_FRAC==1: return df
 # Stratificirano po ispitaniku; uzorak je determinističan za svaki seed.
 return (df.groupby('subject_id',group_keys=False)
         .sample(frac=TRAIN_SAMPLE_FRAC,random_state=seed).sort_index().reset_index(drop=True))

def _finite(x):
 try:
  x=float(x)
  return x if np.isfinite(x) else None
 except (ValueError,TypeError): return None

def write_seed_summary(out,config,seed,pred):
 per=pred.groupby(pred.subject_id.astype(str)).angle_deg.mean().to_dict() if not pred.empty else {}
 info={'config':config,'seed':seed,'mean_angle_global':_finite(pred.angle_deg.mean()),
  'std_angle_global':_finite(pred.angle_deg.std()),
  'per_subject_angle':{str(k):_finite(v) for k,v in per.items()},
  'global_iou':{p:_finite(pred[f'iou_{p}'].mean()) for p in PARTS},
  'num_subjects':len(per),'is_mandatory':True}
 (out/'summary.json').write_text(json.dumps(info,indent=2,allow_nan=False),encoding='utf-8')
 return info

# [RAM zaštita] R3/R4: host OOM fallback i oslobađanje batch objekata.
# [F3] Dodan jasan ispis oporavka batcha kada config.json nedostaje; ostala logika nepromijenjena.
def custom_mtl_objects():
 return {'unit_gaze':unit_gaze,'gaze_cosine':gaze_cosine,'mean_angle_error':mean_angle_error,
         'MaskedBCE':MaskedBCE,'PartIoU':PartIoU,
         'MTL>unit_gaze':unit_gaze,'MTL>gaze_cosine':gaze_cosine,
         'MTL>mean_angle_error':mean_angle_error,'MTL>MaskedBCE':MaskedBCE,'MTL>PartIoU':PartIoU}

def predict_test(model,batch,split='test',seed=42):
 seed_everything(seed)  # [M5] Deterministička inferencija.
 # Recovery batch=16 siguran je za inferenciju; training=False znači bez shufflea/augmentacije.
 df=DF[split]; ds=make_dataset(df,split,batch,False,seed)
 rows=[]; idx=0; skipped=0
 # [M4] Redoslijed je identičan df samo ako nema filtera, shuffla ili drop_remainder.
 for images,targets in ds.as_numpy_iterator():
  pred=model(images,training=False)
  gaze=np.asarray(pred['gaze_output'],np.float32); seg=np.asarray(pred['seg'],np.float32)
  gt=np.asarray(targets['gaze_output'],np.float32); masks=np.asarray(targets['seg'],np.float32)
  assert len(gaze)==len(images)==len(gt)==len(masks), 'Nepodudaran broj predikcija i uzoraka u batchu.'
  assert idx+len(gaze)<=len(df), f'Dataset sadrži više uzoraka od CSV-a: {idx+len(gaze)} > {len(df)}'
  for j in range(len(gaze)):
   record=df.iloc[idx+j]; g=gaze[j]; t=gt[j]
   if not np.isfinite(g).all() or np.linalg.norm(g)==0:
    warnings.warn(f'Preskačem nultu/nevaljanu predikciju: {record.filename}'); skipped+=1; continue
   if not np.isfinite(t).all() or np.linalg.norm(t)==0:
    warnings.warn(f'Preskačem nevaljani GT: {record.filename}'); skipped+=1; continue
   g=g/np.linalg.norm(g); t=t/np.linalg.norm(t)
   row={'subject_id':record.subject_id,'video_id':record.video_id,'filename':record.filename,
        'gt_x':t[0],'gt_y':t[1],'gt_z':t[2],'pred_x':g[0],'pred_y':g[1],'pred_z':g[2],
        'angle_deg':float(np.degrees(np.arccos(np.clip(np.dot(g,t),-1.,1.))))}
   for k,part in enumerate(PARTS):
    valid=masks[j,0,0,k+3]>.5
    a=seg[j,:,:,k]>=.5; b=masks[j,:,:,k]>=.5
    union=np.logical_or(a,b).sum()
    row['iou_'+part]=float(np.logical_and(a,b).sum()/union) if valid and union else (1. if valid else np.nan)
   rows.append(row)
  idx+=len(gaze)
  assert len(rows)<=len(df), 'Broj redaka predikcija premašio je broj redaka CSV-a.'
  # [R4] Oslobodi reference na velike tenzore/nizove nakon svakog batcha.
  del pred, gaze, seg, gt, masks, images, targets
 assert idx==len(df), f'Dataset nije potpun ili je filtriran/drop_remainder: obrađeno {idx}, CSV {len(df)}'
 assert len(rows)+skipped==idx, 'Broj predikcija, preskočenih uzoraka i redaka nije usklađen.'
 return pd.DataFrame(rows)

def train_one(config,seed,parts,w_seg):
 # Očisti grafove i Python objekte prije izgradnje modela za novi seed.
 keras.backend.clear_session(); gc.collect()
 seed_everything(seed)
 out=OUT/config/f'seed_{seed}'; out.mkdir(parents=True,exist_ok=True)
 final=out/'best_model.keras'; weights_final=out/'best_model.weights.h5'
 config_file=out/'config.json'; start=time.monotonic()
 model=None; history=pd.DataFrame(); batch=None
 base_config={'CONFIG_NAME':config,'SEED':seed,'SEG_PARTS_TO_USE':parts,
  'W_SEG':w_seg,'W_GAZE':W_GAZE,'batch_size':None,
  'phase1_epochs':PHASE1_EPOCHS,'phase2_epochs':PHASE2_EPOCHS,
  'TRAIN_SAMPLE_FRAC':TRAIN_SAMPLE_FRAC,'UNFROZEN_LAYERS_PHASE2':UNFROZEN_LAYERS_PHASE2,
  'VAL_STEPS_LIMIT':VAL_STEPS_LIMIT,'is_mandatory':True}
 # Kreiraj config prije provjere postojećeg modela, ali nikada ne prepiši
 # postojeći zapis sa stvarnim batch_size ili informacijama o recoveryju.
 if not config_file.exists():
  config_file.write_text(json.dumps(base_config,indent=2),encoding='utf-8')
 saved_config=json.loads(config_file.read_text(encoding='utf-8'))
 if final.exists() or weights_final.exists():
  print(f'config {config} seed {seed} već postoji, preskačem treniranje')
  if final.exists() and (out/'test_predictions.csv').exists() and (out/'summary.json').exists():
   print('Potpun run već postoji: preskačem učitavanje modela i evaluaciju')
   return None,history,pd.read_csv(out/'test_predictions.csv')
  if final.exists():
   model=keras.models.load_model(final,compile=False,custom_objects=custom_mtl_objects())
  else:
   # Kod fallbacka samo težine nisu samostalno učitljiv model.
   # Ponovno izgradi identičnu arhitekturu pa učitaj težine.
   model,_=build_mtl(seed)
   model.load_weights(str(weights_final))
  if (out/'history.csv').exists(): history=pd.read_csv(out/'history.csv')
  if saved_config.get('batch_size') is not None:
   batch=int(saved_config['batch_size'])
  else:
   warnings.warn(f'{config_file}: batch_size nije poznat; evaluacija s batch=16.')
   batch=16
   saved_config.update(batch_size=batch,batch_recovered=True)
   config_file.write_text(json.dumps(saved_config,indent=2),encoding='utf-8')
 else:
  for candidate in BATCH_FALLBACKS:
   try:
    keras.backend.clear_session(); gc.collect(); seed_everything(seed)
    model,encoder=build_mtl(seed)
    train=make_dataset(sample_train(DF['train'],seed),'train',candidate,True,seed)
    val=make_dataset(DF['val'],'val',candidate).take(VAL_STEPS_LIMIT)
    encoder.trainable=False
    compile_model(model,parts,w_seg,1)
    h1=model.fit(train,validation_data=val,epochs=PHASE1_EPOCHS,callbacks=callbacks(out,1),verbose=2)
    encoder.trainable=True
    for layer in encoder.layers[:-UNFROZEN_LAYERS_PHASE2]: layer.trainable=False
    for layer in encoder.layers[-UNFROZEN_LAYERS_PHASE2:]:
     layer.trainable=not isinstance(layer,layers.BatchNormalization)
    compile_model(model,parts,w_seg,2)
    h2=model.fit(train,validation_data=val,epochs=PHASE2_EPOCHS,callbacks=callbacks(out,2),verbose=2)
    history=pd.concat([pd.DataFrame(h1.history),pd.DataFrame(h2.history)],ignore_index=True)
    # Provjeri sve težine prije spremanja; ne smije se spremiti NaN/Inf model.
    for variable in model.weights:
     if not np.isfinite(variable.numpy()).all():
      raise ValueError(f'Nevaljane (NaN/Inf) težine u {variable.name}; model se ne sprema.')
    # Keras 2.x + mixed precision LossScaleOptimizer ponekad pri serijalizaciji
    # optimizatora pristupa nepostojećem atributu _variables.
    try:
     model.save(final)
    except AttributeError as e:
     if '_variables' not in str(e): raise
     warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
                   'Fallback na spremanje samo težina.')
     # Ne ostavljaj djelomično zapisani .keras koji bi se pogrešno učitao.
     if final.exists(): final.unlink()
     model.save_weights(str(weights_final))
     print(f'[FALLBACK] Težine spremljene u {weights_final}; '
           'pri nastavku će se rekonstruirati arhitektura.')
    else:
     # Provjeri da se kompletan .keras model može ponovno učitati.
     try:
      checked=keras.models.load_model(final,compile=False,custom_objects=custom_mtl_objects())
      del checked
     except Exception as e:
      raise RuntimeError(f'Round-trip load_model NIJE uspio za {final}: {e}') from e
    if hasattr(model,'export'):
     try: model.export(str(out/'best_model_exported'))
     except Exception as e: warnings.warn(f'SavedModel export nije uspio: {e}')
    else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')
    batch=candidate
    break
   except (tf.errors.ResourceExhaustedError,tf.errors.InternalError,MemoryError) as e:
    msg=str(e).upper()
    is_gpu_oom=isinstance(e,tf.errors.ResourceExhaustedError) or 'OOM' in msg
    is_host_oom=isinstance(e,MemoryError) or 'HOST MEMORY' in msg or 'FAILED TO ALLOCATE' in msg
    if not (is_gpu_oom or is_host_oom): raise
    kind='host RAM' if is_host_oom else 'GPU'
    warnings.warn(f'{kind} OOM s batch={candidate}; pokušavam manji batch')
    model=None
    keras.backend.clear_session(); gc.collect()
    if _HAS_PSUTIL:
     try:
      ram=psutil.virtual_memory()
      print(f'  RAM nakon OOM: {ram.percent:.1f}% zauzeto, '
            f'slobodno {ram.available/2**30:.2f} GB')
     except Exception: pass
  if batch is None: raise RuntimeError('OOM i s batch=16')
  history.to_csv(out/'history.csv',index=False)
  saved_config.update(base_config)
  saved_config.update({'batch_size':batch,'phase1_lr':1e-3,'phase2_lr':1e-4,
   'weight_decay':1e-3,
   'augmentation':'baseline flip/rotation/shear/zoom/brightness/contrast',
   'elapsed_seconds':time.monotonic()-start})
  config_file.write_text(json.dumps(saved_config,indent=2),encoding='utf-8')
 pred=predict_test(model,batch,'test',seed)
 pred['is_mandatory_config']=True
 pred.to_csv(out/'test_predictions.csv',index=False)
 val_pred=predict_test(model,batch,'val',seed)
 def fmt(v): return f'{v:.6f}' if np.isfinite(v) else 'NaN'
 seconds=int(time.monotonic()-start); hours,rem=divmod(seconds,3600); minutes,seconds=divmod(rem,60)
 print(f'=== config={config} seed={seed} ===')
 print(f'val_mean_angle_error: {fmt(val_pred.angle_deg.mean())}°')
 print('  '.join(f'val_iou_{p}: {fmt(val_pred["iou_"+p].mean())}' for p in PARTS))
 print(f'trajanje: {hours:02d}:{minutes:02d}:{seconds:02d}')
 if time.monotonic()-start>6*3600: warnings.warn(f'{config}/{seed}: treniranje/evaluacija dulje od 6 h')
 write_seed_summary(out,config,seed,pred)
 return model,history,pred


## 7. Agregacija i MANDATORY petlja

In [7]:
def aggregate():
 mandatory={}; rows=[]
 for conf in MANDATORY_CONFIGS:
  name=conf['name']; per=[]; per_subject={}
  for seed in SEEDS_MANDATORY:
   path=OUT/name/f'seed_{seed}'/'test_predictions.csv'
   if not path.exists(): continue
   df=pd.read_csv(path)
   if df.empty: continue
   subjects=df.groupby(df.subject_id.astype(str)).angle_deg.mean()
   per.append({'seed':seed,'mean_angle':_finite(df.angle_deg.mean()),
    'subject':subjects,'iou':{p:_finite(df[f'iou_{p}'].mean()) if f'iou_{p}' in df else None for p in PARTS}})
   rows.append({'config':name,'seed':seed,'mean_angle':_finite(df.angle_deg.mean()),
    'std_angle':_finite(df.angle_deg.std()),'num_subjects':len(subjects),
    **{f'iou_{p}':per[-1]['iou'][p] for p in PARTS},'is_mandatory':True})
   seed_summary=OUT/name/f'seed_{seed}'/'summary.json'
   if not seed_summary.exists(): write_seed_summary(seed_summary.parent,name,seed,df)
   for sub,val in subjects.items(): per_subject.setdefault(sub,[]).append(float(val))
  if per:
   vals=[x['mean_angle'] for x in per if x['mean_angle'] is not None]
   mandatory[name]={'mean_angle_mean':_finite(np.mean(vals)) if vals else None,
    'mean_angle_std':_finite(np.std(vals,ddof=1)) if len(vals)>1 else 0.,
    'per_subject_mean':{k:_finite(np.mean(v)) for k,v in per_subject.items()},
    'iou_mean':{p:_finite(np.mean([x['iou'][p] for x in per if x['iou'][p] is not None]))
      if any(x['iou'][p] is not None for x in per) else None for p in PARTS},
    'seeds':[x['seed'] for x in per]}
 summary={'mandatory':mandatory,'run_mode':RUN_MODE,'expected_seeds':SEEDS_MANDATORY}
 (OUT/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
 pd.DataFrame(rows,columns=['config','seed','mean_angle','std_angle','num_subjects',
  *[f'iou_{p}' for p in PARTS],'is_mandatory']).to_csv(OUT/'summary.csv',index=False)
 return summary


In [8]:
# Pokreći samo MANDATORY_CONFIGS, bez dodatnih ablacija.
for conf in ACTIVE_CONFIGS:
 for seed in SEEDS_MANDATORY:
  name=conf['name']
  print(f'\n[MANDATORY] {name} seed={seed}')
  model=None
  try:
   model,hist,pred=train_one(name,seed,conf['seg_parts'],conf['w_seg'])
  except KeyboardInterrupt:
   aggregate()
   raise
  except Exception as e:
   warnings.warn(f'{name} seed={seed} nije uspio: {e}')
   traceback.print_exc()
  finally:
   del model
   keras.backend.clear_session(); gc.collect()
   aggregate() # inkrementalno nakon svakog para
print('Obavezna petlja dovršena.')



[MANDATORY] mtl_full seed=42
BN determinism: rekonstruiran encoder s non-fused BatchNormalization; težine sačuvane.
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Epoch 1/5
Vrijeme epohe 3.4 min; prosjek 3.4 min
RAM: 9.2% zauzeto, slobodno 56.99 GB
1327/1327 - 202s - loss: 0.0091 - gaze_output_loss: 0.0053 - seg_loss: 0.0189 - gaze_output_mean_angle_error: 4.1444 - seg_iou_pupil: 0.9144 - seg_iou_iris: 0.9500 - seg_iou_lid: 0.9465 - val_loss: 0.0049 - val_gaze_output_loss: 0.0016 - val_seg_loss: 0.0165 - val_gaze_output_mean_angle_error: 2.7440 - val_seg_iou_pupil: 0.9289 - val_seg_iou_iris: 0.9432 - val_seg_iou_lid: 0.9362 - lr: 0.0010 - 202s/epoch - 152ms/step
Epoch 2/5
Vrijeme epohe 3.2 min; prosjek 3.3 min
RAM: 9.3% zauzeto, sl

/tmp/ipykernel_371991/3638944932.py:133: UserWarning: model.save() nije uspio zbog LossScaleOptimizer _variables: 'LossScaleOptimizerV3' object has no attribute '_variables'. Fallback na spremanje samo težina.
  warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
/tmp/ipykernel_371991/3638944932.py:150: UserWarning: model.export() nije dostupan; preskačem SavedModel export.
  else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')


[FALLBACK] Težine spremljene u /home/fipulab/anastazija/runs/mtl/mtl_full/seed_42/best_model.weights.h5; pri nastavku će se rekonstruirati arhitektura.
=== config=mtl_full seed=42 ===
val_mean_angle_error: 2.761213°
val_iou_pupil: 0.917242  val_iou_iris: 0.927526  val_iou_lid: 0.906765
trajanje: 00:44:49

[MANDATORY] mtl_full seed=123
BN determinism: rekonstruiran encoder s non-fused BatchNormalization; težine sačuvane.
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
Epoch 1/5
Vrijeme epohe 3.3 min; prosjek 3.3 min
RAM: 10.1% zauzeto, slobodno 56.40 GB
1327/1327 - 196s - loss: 0.0093 - gaze_output_loss: 0.0052 - seg_loss: 0.0205 - gaze_output_mean_angle_error: 3.9867 - seg_iou_pupil: 0.9121 - seg_iou_iris: 0.9465 - seg_iou_lid: 0.9428 - val_loss: 0.0050 - val_gaze_output_loss: 0.0017 - val_seg_loss: 0.0168 - val_gaze_output_mean_angle_error: 2.6760 - val_seg_iou_pupil: 0.9221 - val_seg_iou_iris: 0.9469 - val_seg_iou_lid: 0.9340 - 

/tmp/ipykernel_371991/3638944932.py:133: UserWarning: model.save() nije uspio zbog LossScaleOptimizer _variables: 'LossScaleOptimizerV3' object has no attribute '_variables'. Fallback na spremanje samo težina.
  warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
/tmp/ipykernel_371991/3638944932.py:150: UserWarning: model.export() nije dostupan; preskačem SavedModel export.
  else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')


=== config=mtl_full seed=123 ===
val_mean_angle_error: 3.132229°
val_iou_pupil: 0.919221  val_iou_iris: 0.926201  val_iou_lid: 0.907146
trajanje: 00:44:18

[MANDATORY] mtl_full seed=2024
BN determinism: rekonstruiran encoder s non-fused BatchNormalization; težine sačuvane.
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
Epoch 1/5
Vrijeme epohe 3.2 min; prosjek 3.2 min
RAM: 10.3% zauzeto, slobodno 56.28 GB
1327/1327 - 192s - loss: 0.0115 - gaze_output_loss: 0.0076 - seg_loss: 0.0196 - gaze_output_mean_angle_error: 4.2696 - seg_iou_pupil: 0.9128 - seg_iou_iris: 0.9490 - seg_iou_lid: 0.9464 - val_loss: 0.0047 - val_gaze_output_loss: 0.0016 - val_seg_loss: 0.0152 - val_gaze_output_mean_angle_error: 2.7805 - val_seg_iou_pupil: 0.9205 - val_seg_iou_iris: 0.9519 - val_seg_iou_lid: 0.9410 - lr: 0.0010 - 192s/epoch - 144ms/step
Epoch 2/5
Vrijeme epohe 3.2 min; prosjek 3.2 min
RAM: 10.3% zauzeto, slobodno 56.28 GB
1327/1327 - 191s - loss: 0

/tmp/ipykernel_371991/3638944932.py:133: UserWarning: model.save() nije uspio zbog LossScaleOptimizer _variables: 'LossScaleOptimizerV3' object has no attribute '_variables'. Fallback na spremanje samo težina.
  warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
/tmp/ipykernel_371991/3638944932.py:150: UserWarning: model.export() nije dostupan; preskačem SavedModel export.
  else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')


... (10 outputs left out)

/tmp/ipykernel_371991/3638944932.py:133: UserWarning: model.save() nije uspio zbog LossScaleOptimizer _variables: 'LossScaleOptimizerV3' object has no attribute '_variables'. Fallback na spremanje samo težina.
  warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
/tmp/ipykernel_371991/3638944932.py:150: UserWarning: model.export() nije dostupan; preskačem SavedModel export.
  else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')


[FALLBACK] Težine spremljene u /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_123/best_model.weights.h5; pri nastavku će se rekonstruirati arhitektura.
=== config=mtl_pupil seed=123 ===
val_mean_angle_error: 2.948795°
val_iou_pupil: 0.927606  val_iou_iris: 0.202472  val_iou_lid: 0.465543
trajanje: 00:56:47

[MANDATORY] mtl_pupil seed=2024
BN determinism: rekonstruiran encoder s non-fused BatchNormalization; težine sačuvane.
decoder_features: resizing_2 (None, 128, 128, 12)
encoder_features: feature_extractor (None, 4, 4, 576)
Epoch 1/5
Vrijeme epohe 3.2 min; prosjek 3.2 min
RAM: 11.1% zauzeto, slobodno 55.77 GB
1327/1327 - 190s - loss: 0.0020 - gaze_output_loss: 0.0013 - seg_loss: 0.0036 - gaze_output_mean_angle_error: 2.2559 - seg_iou_pupil: 0.9242 - seg_iou_iris: 0.2488 - seg_iou_lid: 0.5700 - val_loss: 0.0019 - val_gaze_output_loss: 0.0015 - val_seg_loss: 0.0020 - val_gaze_output_mean_angle_error: 2.8074 - val_seg_iou_pupil: 0.9346 - val_seg_iou_iris: 0.2201 - val_seg_iou_lid: 0.5

/tmp/ipykernel_371991/3638944932.py:133: UserWarning: model.save() nije uspio zbog LossScaleOptimizer _variables: 'LossScaleOptimizerV3' object has no attribute '_variables'. Fallback na spremanje samo težina.
  warnings.warn(f'model.save() nije uspio zbog LossScaleOptimizer _variables: {e}. '
/tmp/ipykernel_371991/3638944932.py:150: UserWarning: model.export() nije dostupan; preskačem SavedModel export.
  else: warnings.warn('model.export() nije dostupan; preskačem SavedModel export.')


[FALLBACK] Težine spremljene u /home/fipulab/anastazija/runs/mtl/mtl_pupil/seed_2024/best_model.weights.h5; pri nastavku će se rekonstruirati arhitektura.
=== config=mtl_pupil seed=2024 ===
val_mean_angle_error: 2.994805°
val_iou_pupil: 0.925425  val_iou_iris: 0.277580  val_iou_lid: 0.600455
trajanje: 00:44:21
Obavezna petlja dovršena.


## 8. Wilcoxon po ispitaniku i seedu

In [9]:
def subject_means(df):
 if 'subject_id' not in df or 'angle_deg' not in df:
  raise ValueError('Predikcije moraju sadržavati subject_id i angle_deg')
 return df.groupby(df.subject_id.astype(str)).angle_deg.mean()

def paired_test(a,b):
 a=subject_means(a); b=subject_means(b)
 common=a.index.intersection(b.index)
 x=a.loc[common].to_numpy(float); y=b.loc[common].to_numpy(float)
 good=np.isfinite(x)&np.isfinite(y); x=x[good]; y=y[good]
 if len(x)<2: return None,None,len(x)
 if np.allclose(x-y,0,rtol=0,atol=0): return 0.,1.,len(x)
 result=wilcoxon(x,y,zero_method='wilcox')
 return _finite(result.statistic),_finite(result.pvalue),len(x)

# Seed-level uparivanje: subject_id je jedinica testa; nikad pojedinačni frame.
# Zajednički rezultat preko seedova: za svakog ispitanika prosjek preko
# dostupnih uparenih seedova, zatim jedan Wilcoxon po ispitaniku.
def compare_runs(label,paths_a,paths_b):
 records=[]; per_subject={}
 for seed in SEEDS_MANDATORY:
  a=paths_a(seed); b=paths_b(seed)
  if not a.exists() or not b.exists():
   warnings.warn(f'Wilcoxon {label} seed={seed}: nedostaje {a if not a.exists() else b}')
   continue
  try:
   da=pd.read_csv(a); db=pd.read_csv(b)
   stat,p,n=paired_test(da,db)
   records.append({'comparison':label,'seed':seed,'statistic':stat,'p_value':p,'num_subjects':n,'level':'per_seed'})
   sa=subject_means(da); sb=subject_means(db)
   for subject in sa.index.intersection(sb.index):
    x,y=sa.loc[subject],sb.loc[subject]
    if np.isfinite(x) and np.isfinite(y): per_subject.setdefault(subject,[]).append((float(x),float(y)))
  except Exception as e: warnings.warn(f'Wilcoxon {label} seed={seed}: {e}')
 if per_subject:
  aa=pd.DataFrame({'subject_id':list(per_subject),'angle_deg':[np.mean([z[0] for z in v]) for v in per_subject.values()]})
  bb=pd.DataFrame({'subject_id':list(per_subject),'angle_deg':[np.mean([z[1] for z in v]) for v in per_subject.values()]})
  stat,p,n=paired_test(aa,bb)
  records.append({'comparison':label,'seed':'pooled_subject','statistic':stat,'p_value':p,'num_subjects':n,'level':'across_seeds'})
 return records

wilcox=[]
for conf in MANDATORY_CONFIGS:
 name=conf['name']
 wilcox+=compare_runs(f'{name} vs gaze_only',
  lambda s,n=name: OUT/n/f'seed_{s}'/'test_predictions.csv',
  lambda s: RUNS/'gaze_only'/f'seed_{s}'/'test_predictions.csv')
wilcox+=compare_runs('mtl_wseg0 vs mtl_full',
 lambda s: OUT/'mtl_wseg0'/f'seed_{s}'/'test_predictions.csv',
 lambda s: OUT/'mtl_full'/f'seed_{s}'/'test_predictions.csv')
trivial_path=RUNS/'trivial_baseline'/'test_predictions.csv'
if trivial_path.exists():
 wilcox+=compare_runs('trivial_baseline vs gaze_only',lambda s: trivial_path,
  lambda s: RUNS/'gaze_only'/f'seed_{s}'/'test_predictions.csv')
else: warnings.warn('Trivijalni baseline nedostaje: preskačem njegov Wilcoxon')
pd.DataFrame(wilcox,columns=['comparison','seed','statistic','p_value','num_subjects','level']).to_csv(OUT/'wilcoxon.csv',index=False)
print('Wilcoxon:',len(wilcox),'redaka')


Wilcoxon: 16 redaka


/tmp/ipykernel_371991/330965218.py:55: UserWarning: Trivijalni baseline nedostaje: preskačem njegov Wilcoxon
  else: warnings.warn('Trivijalni baseline nedostaje: preskačem njegov Wilcoxon')


## 9–10. Završni summary i vizualizacije

In [10]:
summary=aggregate()
figdir=OUT/'figures'; figdir.mkdir(parents=True,exist_ok=True)
values=[]; names=[]; errors=[]
for name,data in summary['mandatory'].items():
 if data['mean_angle_mean'] is not None:
  names.append(name);values.append(data['mean_angle_mean']);errors.append(data['mean_angle_std'] or 0.)
for label,paths in [('gaze_only',[RUNS/'gaze_only'/f'seed_{s}'/'test_predictions.csv' for s in SEEDS_MANDATORY]),
                    ('trivial_baseline',[trivial_path])]:
 vals=[]
 for p in paths:
  if p.exists():
   try: vals.append(pd.read_csv(p).angle_deg.mean())
   except Exception as e: warnings.warn(f'{p}: {e}')
 if vals:
  names.append(label);values.append(float(np.mean(vals)));errors.append(float(np.std(vals,ddof=1)) if len(vals)>1 else 0.)
if values:
 plt.figure(figsize=(max(9,len(names)*1.7),5))
 plt.bar(names,values,yerr=errors,capsize=4)
 plt.ylabel('Mean angle error (°)');plt.xticks(rotation=25,ha='right');plt.tight_layout()
 plt.savefig(figdir/'angle_by_config.png',dpi=180);plt.close()
boxes=[];labels=[]
for name in ('mtl_full','mtl_wseg0','gaze_only'):
 for seed in SEEDS_MANDATORY:
  p=(RUNS/'gaze_only' if name=='gaze_only' else OUT/name)/f'seed_{seed}'/'test_predictions.csv'
  if p.exists():
   try:
    x=subject_means(pd.read_csv(p)).dropna().to_numpy()
    if len(x):boxes.append(x);labels.append(f'{name} {seed}')
   except Exception as e:warnings.warn(f'Boxplot {p}: {e}')
if boxes:
 plt.figure(figsize=(max(10,len(boxes)*1.35),5))
 try:plt.boxplot(boxes,tick_labels=labels)
 except TypeError:plt.boxplot(boxes,labels=labels)
 plt.ylabel('Mean angle error po ispitaniku (°)');plt.xticks(rotation=50,ha='right');plt.tight_layout()
 plt.savefig(figdir/'subject_boxplot.png',dpi=180);plt.close()
print('Rezultati:',OUT/'summary.json',OUT/'summary.csv',OUT/'wilcoxon.csv',figdir)


Rezultati: /home/fipulab/anastazija/runs/mtl/summary.json /home/fipulab/anastazija/runs/mtl/summary.csv /home/fipulab/anastazija/runs/mtl/wilcoxon.csv /home/fipulab/anastazija/runs/mtl/figures
